In [1]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model'
os.chdir(workdir)
os.getcwd()

In [2]:
import copy

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl

import vcfparser as vcfp
import model.bndbpconv as bndc

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
# We need at least 1 path to population reference and 1 path to a callset
ref_path = 'data/gnomad.v4.1.sv.sites.vcf.gz'
callset_path = 'data/test_gridss_output.NA12878.vcf'

In [ ]:
callset_df = vcfp.parse_vcf(callset_path)

In [34]:
callset_events = bndc.bnd_to_event(callset_df)

In [35]:
callset_events

In [9]:
test_callset_df = test_callset_df.sort_values(by=['CHROM', 'POS'])

In [ ]:
def bnd_to_event(vcf_df):
    """
    Converts a dataframe with breakends only to dataframe with events.
    CURRENTLY ONLY WORKS WITH DELS
    """
    
    vcf_df_copy = vcf_df[test_callset_df['ALT'].str.contains('|'.join(['\\[', '\\]']))].copy()

    cols_for_retaining = get_cols_for_retaining(vcf_df_copy)

    group_dfs = get_group_dfs(vcf_df_copy.loc[:, cols_for_retaining])
    del vcf_df_copy

    group_df_total = merge_group_lists(group_dfs)

    group_df_total = group_df_total[
        (group_df_total['CHROM'] == group_df_total['CHR2']) & \
    
        (
    # 2 Ways a DEL could be recorded:
            (
                (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
                (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
                (group_df_total['INFO_END'] < group_df_total['POS'])
            ) | \
            (
                (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
                (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
                (group_df_total['POS'] < group_df_total['INFO_END'])
            )
        )
    ]

    group_df_del_only = group_df_del_only[
        (
            group_df_del_only['LALT'].str.contains('^[ATGCNatgcn]{1}\\[', ) & \
            group_df_del_only['RALT'].str.contains('\\][ATGCNatgcn]$', )
        ) | \
        (
            group_df_del_only['RALT'].str.contains('^[ATGCNatgcn]{1}\\[', ) & \
            group_df_del_only['LALT'].str.contains('\\][ATGCNatgcn]$', )
        )
    ]
    
    group_df_del_only.loc[:, 'REF'] = 'N'
    group_df_del_only.loc[:, 'ALT'] = '<DEL>'
    group_df_del_only.loc[:, 'INFO_SVTYPE'] = 'DEL'

    return group_df_del_only

In [10]:
# Retain only breakends with a pair
test_callset_df = test_callset_df[test_callset_df['ALT'].str.contains('|'.join(['\\[', '\\]']))]

In [13]:
cols_for_retaining = get_cols_for_retaining(test_callset_df)

In [15]:
group_dfs = get_group_dfs(test_callset_df.loc[:, cols_for_retaining])

In [17]:
group_df_total = merge_group_lists(group_dfs)

In [18]:
group_df_total

In [ ]:
# group_df_total = pd.concat(
#     [
#         pd.DataFrame(group_list1).reset_index(drop=True),
#         pd.DataFrame(group_list2).reset_index(drop=True)
#     ],
#     axis=1
# )

In [ ]:
group_df_total[
    (group_df_total['CHROM'] == group_df_total['CHR2']) & \

    (
# 2 Ways a DEL could be recorded:
        (
            (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
            (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
            (group_df_total['INFO_END'] < group_df_total['POS'])
        ) | \
        (
            (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
            (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
            (group_df_total['POS'] < group_df_total['INFO_END'])
        )
    )
]

In [19]:
# AFTER GROUP LIST IS DONE WITH FILTERING - REDO

group_df_del_only = group_df_total[
    (group_df_total['CHROM'] == group_df_total['CHR2']) & \

    (
# 2 Ways a DEL could be recorded:
        (
            (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
            (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
            (group_df_total['INFO_END'] < group_df_total['POS'])
        ) | \
        (
            (group_df_total['LALT'].str.strip('ATGCNatgcn').str[0] == '[') & \
            (group_df_total['RALT'].str.strip('ATGCNatgcn').str[0] == ']') & \
            (group_df_total['POS'] < group_df_total['INFO_END'])
        )
    )
]
len(group_df_del_only)

In [20]:
group_df_del_only = group_df_del_only[
    (
        group_df_del_only['LALT'].str.contains('^[ATGCNatgcn]{1}\\[', ) & \
        group_df_del_only['RALT'].str.contains('\\][ATGCNatgcn]$', )
    ) | \
    (
        group_df_del_only['RALT'].str.contains('^[ATGCNatgcn]{1}\\[', ) & \
        group_df_del_only['LALT'].str.contains('\\][ATGCNatgcn]$', )
    )
]


In [21]:
group_df_del_only

In [24]:
group_df_del_only.loc[:, 'REF'] = 'N'
group_df_del_only.loc[:, 'ALT'] = '<DEL>'
group_df_del_only.loc[:, 'INFO_SVTYPE'] = 'DEL'

In [25]:
group_df_del_only